# 01 — Ingestão e limpeza dos dados

_Notebook do projeto **crime-sp-ml**. Reaproveita os módulos em `src/`._

In [1]:
import sys, pathlib
from IPython.display import Image, IFrame, display, Markdown
ROOT = pathlib.Path.cwd()
while not (ROOT/'src').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT/'src'))
FIG = ROOT/'reports'/'figures'; MAPS = ROOT/'reports'/'maps'; REP = ROOT/'reports'
print('Projeto:', ROOT.name)

Projeto: crime-sp-ml


Carga do dataset processado da SSP/SP (gerado por `src/pipeline.py`) e a limpeza final de `src/data_loader.py`: filtro geográfico, parsing de hora, normalização de período e correção do alvo.

In [2]:
import data_loader as dl
df = dl.load_clean()
print('linhas:', f'{len(df):,}')
df.head(3)

linhas: 2,084,529


,DATA_OCORRENCIA_BO,HORA_OCORRENCIA_BO,DESCR_TIPOLOCAL,DESCR_SUBTIPOLOCAL,BAIRRO,LOGRADOURO,NUMERO_LOGRADOURO,LATITUDE,LONGITUDE,DESCR_CONDUTA,...,lat,lon,geo_valid,hora,minuto,tem_hora_exata,periodo_texto,crime,data,ano
0,2024-12-31 00:00:00,18:00:00,Comércio e Serviços,Mercado,LIBERDADE,VEDACAO DA DIVULGACAO DOS DADOS RELATIVOS,0,0,0,NaN,...,0.0,0.0,False,18,0,True,NaN,ESTUPRO DE VULNERAVEL,2024-12-31,2024
1,2025-01-10 00:00:00,00:00:00,Via Pública,Via Pública,SE,VEDACAO DA DIVULGACAO DOS DADOS RELATIVOS,0,0,0,NaN,...,0.0,0.0,False,0,0,True,NaN,ESTUPRO DE VULNERAVEL,2025-01-10,2025
2,2025-01-14 00:00:00,NaN,Residência,Casa,SE,VEDACAO DA DIVULGACAO DOS DADOS RELATIVOS,0,0,0,NaN,...,0.0,0.0,False,<NA>,<NA>,False,indeterminado,ESTUPRO DE VULNERAVEL,2025-01-14,2025


### Resumo do dataset (após limpeza)

In [3]:
import json
print(json.dumps(dl.resumo_dataset(df), indent=2, ensure_ascii=False))

{
  "linhas_totais": 2084529,
  "geo_validas": 1885004,
  "com_hora_exata": 1379343,
  "linhas_supervisionado": 1241212,
  "classes_supervisionado": 20,
  "linhas_exploratorio": 1805961,
  "classes_totais": 24
}


### Distribuição do alvo `NATUREZA_APURADA`

In [4]:
df['crime'].value_counts().head(15)

crime
FURTO - OUTROS                                     1058041
ROUBO - OUTROS                                      509769
FURTO DE VEICULO                                    175311
LESAO CORPORAL DOLOSA                               156804
LESAO CORPORAL CULPOSA POR ACIDENTE DE TRANSITO      58659
ROUBO DE VEICULO                                     56262
TRAFICO DE ENTORPECENTES                             23313
ROUBO DE CARGA                                       10299
ESTUPRO DE VULNERAVEL                                 9545
PORTE DE ENTORPECENTES                                3945
PORTE DE ARMA                                         3554
APREENSAO DE ENTORPECENTES                            3434
ESTUPRO                                               3235
LESAO CORPORAL CULPOSA - OUTRAS                       3219
TENTATIVA DE HOMICIDIO                                2936
Name: count, dtype: Int64

**Observações:** coordenadas inválidas (lat/long=0) e 'hora incerta' são tratadas; o alvo tem 24 classes, fortemente desbalanceadas.